Task 1: Build a Robust NLP Preprocessing Engine

Task 1: Conceptual Understanding

1. Difference between "Love" and "love" in NLP

ANS -> In NLP, text is often case-sensitive. This means "Love" and "love" are treated as two different words (tokens).
For example, a model might count them separately, which can create unnecessary duplication.
To avoid this, we usually convert all text to lowercase during preprocessing so both become the same token: "love".

2. What happens if stopwords are not removed?

 Ans -> Stopwords are common words like is, the, and, in.
If they are not removed:

The dataset becomes noisy and less meaningful
Models may focus on irrelevant words instead of important ones
It can increase computation time and reduce efficiency

However, removing them is not always necessary depending on the task.

  3. Two real-world scenarios where removing stopwords can be harmful
Ans->

1-> Sentiment Analysis
Words like "not" are very important.
Example:

"This is good" → positive
"This is not good" → negative
If we remove "not", the meaning becomes incorrect.

2-> Chatbots / Question Answering Systems
Stopwords help maintain sentence structure and meaning.
Example:

"Where is the nearest hospital?"
Removing stopwords may confuse the intent and reduce accuracy.

4. Difference between stemming and lemmatization

Stemming

Cuts words to their base form using simple rules
May produce incorrect or incomplete words
Example: running → runn

Lemmatization

Converts words into their correct dictionary form
Uses vocabulary and grammar rules
Example: running → run

In [ ]:
import re

def preprocess_text(text):
    """
    Preprocess raw text into clean tokens and sentence.

    Steps:
    - Handle empty/invalid input
    - Convert to lowercase
    - Remove URLs and emails
    - Remove numbers
    - Normalize repeated characters
    - Remove special characters / punctuation
    - Remove extra spaces
    - Tokenize
    - Remove short tokens (<=2 except 'no', 'not')
    """

    # Error handling
    if not isinstance(text, str) or text.strip() == "":
        return [], ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove email patterns
    text = re.sub(r'\S+@\S+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Handle repeated characters (soooo → soo)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Remove special characters / punctuation / emojis
    text = re.sub(r'[^\w\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = text.split()

    # Remove short tokens (≤2), keep 'no' and 'not'
    filtered_tokens = [
        word for word in tokens
        if len(word) > 2 or word in ['no', 'not']
    ]

    # Rebuild clean sentence
    clean_sentence = " ".join(filtered_tokens)

    return filtered_tokens, clean_sentence

In [ ]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product ",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

In [ ]:
for i, sentence in enumerate(test_sentences, 1):
    tokens, clean_sentence = preprocess_text(sentence)

    print(f"Sentence {i}")
    print("Original Text   :", sentence)
    print("Cleaned Tokens  :", tokens)
    print("Cleaned Sentence:", clean_sentence)
    print("-" * 60)

Sentence 1
Original Text   : Get 100% FREE access now!!!
Cleaned Tokens  : ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
------------------------------------------------------------
Sentence 2
Original Text   : I absolutely looooved this product 
Cleaned Tokens  : ['absolutely', 'looved', 'this', 'product']
Cleaned Sentence: absolutely looved this product
------------------------------------------------------------
Sentence 3
Original Text   : Worst service ever... 0/10
Cleaned Tokens  : ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
------------------------------------------------------------
Sentence 4
Original Text   : Call me at 9876543210
Cleaned Tokens  : ['call']
Cleaned Sentence: call
------------------------------------------------------------
Sentence 5
Original Text   : This is THE best course!!!
Cleaned Tokens  : ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
----------------------------------------------

In [ ]:
results = []

for sentence in test_sentences:
    tokens, clean = preprocess_text(sentence)
    results.append((sentence, tokens, clean))

for i, (sentence, tokens, clean) in enumerate(results, 1):

    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))

    avg_token_length = 0
    if total_tokens > 0:
        avg_token_length = sum(len(word) for word in tokens) / total_tokens

    print(f"Sentence {i}")
    print("Original Text:", sentence)
    print("Total Tokens:", total_tokens)
    print("Unique Tokens:", unique_tokens)
    print("Average Token Length:", round(avg_token_length, 2))
    print("-" * 60)

Sentence 1
Original Text: Get 100% FREE access now!!!
Total Tokens: 4
Unique Tokens: 4
Average Token Length: 4.0
------------------------------------------------------------
Sentence 2
Original Text: I absolutely looooved this product 
Total Tokens: 4
Unique Tokens: 4
Average Token Length: 6.75
------------------------------------------------------------
Sentence 3
Original Text: Worst service ever... 0/10
Total Tokens: 3
Unique Tokens: 3
Average Token Length: 5.33
------------------------------------------------------------
Sentence 4
Original Text: Call me at 9876543210
Total Tokens: 1
Unique Tokens: 1
Average Token Length: 4.0
------------------------------------------------------------
Sentence 5
Original Text: This is THE best course!!!
Total Tokens: 4
Unique Tokens: 4
Average Token Length: 4.25
------------------------------------------------------------
Sentence 6
Original Text: Visit https://openai.com now!
Total Tokens: 2
Unique Tokens: 2
Average Token Length: 4.0
------------

In [ ]:
from collections import Counter

all_tokens = []

for _, tokens, _ in results:
    all_tokens.extend(tokens)

In [ ]:
word_counts = Counter(all_tokens)

In [ ]:
print("Top 10 Most Frequent Words:")
for word, count in word_counts.most_common(10):
    print(f"{word}: {count}")

Top 10 Most Frequent Words:
this: 4
now: 3
get: 1
free: 1
access: 1
absolutely: 1
looved: 1
product: 1
worst: 1
service: 1


In [ ]:
print("\nTop 5 Least Frequent Words:")

least_common = word_counts.most_common()[::-1][:5]

for word, count in least_common:
    print(f"{word}: {count}")


Top 5 Least Frequent Words:
with: 1
happy: 1
not: 1
offer: 1
limited: 1


In [ ]:
def full_pipeline(text_list):
    """
    Processes a list of raw text inputs and returns:
    - Combined tokens
    - Cleaned sentences

    Args:
        text_list (list): List of raw text strings

    Returns:
        dict: {
            "tokens": [...],
            "clean_sentences": [...]
        }
    """

    all_tokens = []
    clean_sentences = []

    # ⚠️ Handle invalid input
    if not isinstance(text_list, list):
        return {"tokens": [], "clean_sentences": []}

    for text in text_list:
        tokens, clean = preprocess_text(text)

        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

In [ ]:
output = full_pipeline(test_sentences)

print("All Tokens:")
print(output["tokens"])

print("\nClean Sentences:")
for sentence in output["clean_sentences"]:
    print(sentence)

All Tokens:
['get', 'free', 'access', 'now', 'absolutely', 'looved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'noo', 'this', 'baad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this']

Clean Sentences:
get free access now
absolutely looved this product
worst service ever
call
this the best course
visit now
noo this baad
got
win now limited offer
not happy with this


In [ ]:
{
    "tokens": ['get', 'free', 'access', 'now', 'absolutely', 'looved', ...],
    "clean_sentences": [
        "get free access now",
        "absolutely looved this product",
        "worst service ever",
        ...
    ]
}

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'looved', Ellipsis],
 'clean_sentences': ['get free access now',
  'absolutely looved this product',
  'worst service ever',
  Ellipsis]}